# Agents II: Infrastructure and Evaluation


Lecture 21 was about the agent loop and the harness that wraps a model: a
coding agent as six components. This lecture is about everything *around* that
loop once agents leave one developer's laptop: how a model's tool calls become
a protocol that any tool can speak, how an agent remembers across a task that
is longer than its context window, when several agents are better than one and
when they are worse, how the RL of lecture 14 changes when the trajectory
contains tool calls, how to measure an agent at all, and what goes wrong when
an agent reads untrusted text. UDL does not cover agents; the notation follows
the primary papers and each is cited where it is used. The quizzable items
are two calculations, the tokens-per-task sum with and without a prefix cache
and the pass^k reliability metric, and one reasoning exercise, the lethal
trifecta.


## Today's roadmap

1. Recap: the loop from lecture 21, and what this lecture adds around it
2. Tools and protocols: function calling, the Model Context Protocol, A2A
3. Memory and context management: working memory, compaction, retrieval, cost
4. Multi-agent patterns: when more agents help, when they fail, research agents
5. Agentic RL: what changes from lecture 14
6. Evaluating agents: task suites, trajectory grading, pass^k, cost per task
7. Safety and failure modes: prompt injection, the lethal trifecta, permissions
8. Summary


# Recap: the loop, and the parts around it

Two slides of recap so that the rest of the lecture has a shared picture to
point at.


## The agent loop, from lecture 21

An agent is a control loop around a model: **observe** the environment,
**choose** an action, **act** (a tool call, an edit, a command), observe
again, until the goal is met or a budget runs out. The harness around it
supplied six things: live repo context, a cacheable prompt shape, structured
tools with permissions, context reduction, transcripts and working memory, and
bounded subagents.

Three facts from lecture 21 carry over unchanged:

- The model only ever sees **one growing text**: its context. Everything the
  harness does is a decision about what goes into that text.
- A tool call is **structured output** that the harness validates, approves,
  runs and clips before the result re-enters the context.
- Prompt caching makes the loop affordable: stable prefix first, changing
  tail last.


## What this lecture adds: the parts around the model

![The loop of lecture 21 in the middle; this lecture's five sections around it. Source: instructor figure.](figs/22-Agent-Infrastructure-and-Evaluation/agent-loop-and-surroundings.png)

# Tools and protocols

Lecture 21 showed a coding harness with half a dozen hand-written tools. The
first question of infrastructure is what happens when there are thousands of
tools and dozens of agent applications: who writes the glue? The answer since
late 2024 is a protocol.


## A tool call is structured output

The harness describes each tool with a **JSON Schema**; the model emits a JSON
object that matches it; the harness runs the function and returns the result as
a new message. The schema is the whole contract.

**Tool definition (sent in the prompt)**

```json
{"name": "run_tests",
 "description": "Run the test suite; return failures.",
 "input_schema": {
   "type": "object",
   "properties": {
     "path":    {"type": "string"},
     "verbose": {"type": "boolean"}},
   "required": ["path"]}}
```

**Model output (a tool-use turn)**

```json
{"type": "tool_use",
 "name": "run_tests",
 "input": {"path": "tests/test_auth.py"}}
```

**Harness reply (a tool-result turn)**

```json
{"type": "tool_result",
 "content": "FAILED test_login_timeout ..."}
```


Introduced as an API feature by OpenAI in June 2023 with models fine-tuned to
"detect when a function needs to be called and respond with JSON that adheres
to the function signature"
([OpenAI 2023](https://openai.com/index/function-calling-and-other-api-updates));
every provider and every open-weight chat model now has an equivalent.


## How a model learns to emit calls

- **Supervised fine-tuning on tool-call traces.** Llama 3's post-training
  includes synthetic single-turn, multi-turn and multi-step tool-use dialogs,
  with the chat format extended so a turn can address a tool rather than the
  user ([Grattafiori et al. 2024](https://arxiv.org/abs/2407.21783), section 4.3.5).
  This is lecture 13's SFT, with the demonstration being a call.
- **Constrained decoding.** A JSON Schema compiles to a finite-state machine
  over tokens; at each step the sampler masks every token that would leave
  the language, so the output is valid by construction
  ([Willard and Louf 2023](https://arxiv.org/abs/2307.09702)). Constraint
  guarantees the syntax, not the choice of tool or the argument values; RL on
  tool-use tasks (section 5) is what improves those.


## The N × M problem, and what a protocol is for

- Lecture 21's harness hand-wrote `read_file`, `run_shell`, `apply_diff`. Now
  add GitHub, a database, a browser, Slack, your lab's instrument. Each agent
  application (an IDE, a chat app, a CLI harness) would need its own connector
  for each: $N$ applications × $M$ tools of glue code.
- A **protocol** turns that into $N + M$: each tool is wrapped once as a
  *server*; each application implements one *client*. The model never sees the
  protocol; it sees the same tool schemas as before.
- The precedent is the Language Server Protocol, which did this for editors
  and language tooling in 2016. The Model Context Protocol says so explicitly
  ([MCP specification](https://modelcontextprotocol.io/specification/2026-07-28/)).
- The standardization also carries conventions the model benefits from: how a
  tool is described, how its result is typed, how a human is asked for consent.


## The Model Context Protocol: hosts, clients, servers

![The three roles. One host runs the model and the loop; it opens one client per server; servers expose tools, resources and prompts over JSON-RPC. Source: instructor figure after the MCP architecture page.](figs/22-Agent-Infrastructure-and-Evaluation/mcp-host-client-server.png)

- Released by Anthropic in November 2024 as an open standard
  ([Anthropic 2024](https://www.anthropic.com/news/model-context-protocol)).
  The spec names three roles: **hosts** are "LLM applications that initiate
  connections", **clients** are "connectors within the host application",
  **servers** are "services that provide context and capabilities".
- Messages are JSON-RPC 2.0. The host decides which servers to connect, which
  tools to show the model, and when to ask the user; the server never talks to
  the model directly.


## The three server primitives, and one client primitive

| Primitive | Offered by | What it is | Who uses it |
|---|---|---|---|
| **Tools** | server | functions the model may call, each with a JSON Schema | the model |
| **Resources** | server | context and data addressed by URI (a file, a table, a log) | the host or the model, read-only |
| **Prompts** | server | templated messages and workflows | the user, via the host's UI |
| **Elicitation** | client | a server-initiated request for more information from the user | the server, through the host |

Tools are what the model acts with; resources are what it reads; prompts are
pre-packaged instructions. The spec's security principles follow from the
roles: "hosts must obtain explicit user consent before invoking any tool", and
tool descriptions "should be considered untrusted, unless obtained from a
trusted server" ([MCP specification](https://modelcontextprotocol.io/specification/2026-07-28/),
Security and Trust & Safety). Two earlier client primitives, *sampling* (a
server asking the host's model to generate) and *roots* (filesystem
boundaries), were deprecated in the 2026-07-28 revision
([deprecated features registry](https://modelcontextprotocol.io/specification/2026-07-28/deprecated)).


## Transports and versions

Two transports: **stdio**, where the client launches the server as a
subprocess and talks JSON-RPC over its standard input and output (local, zero
network), and **Streamable HTTP**, an independent server behind one HTTP
endpoint ([MCP transports](https://modelcontextprotocol.io/specification/2025-06-18/basic/transports)).
Versions are dated (`2026-07-28` is current); check the spec
([MCP versioning](https://modelcontextprotocol.io/specification/versioning)).

## A minimal MCP server

Illustrative only; the official Python SDK's quickstart, current as of
`mcp` 2.x ([python-sdk](https://github.com/modelcontextprotocol/python-sdk)).
Earlier 1.x releases spelled the class `FastMCP`.

```python
# server.py  --  one tool, one templated resource
# pip install "mcp[cli]"                        # the extra provides the `mcp` CLI used below
from mcp.server import MCPServer

mcp = MCPServer("experiments")

@mcp.tool()
def best_run(metric: str = "val_bpb") -> dict:
    """Return the best logged run by the given metric."""
    rows = ledger.query(metric)            # your own code
    return min(rows, key=lambda r: r[metric])

@mcp.resource("ledger://runs/{run_id}")
def run_log(run_id: str) -> str:
    """Full log of one experiment."""
    return ledger.read_log(run_id)

# uv run mcp dev server.py                      # stdio, in the inspector
# uv run mcp run server.py --transport streamable-http
```

The type hints *are* the JSON Schema; the docstring *is* the tool description
the model reads. Nothing here mentions a model: the server is reusable by any
host.


## What the host does with it

1. **Connect** a client to each configured server and ask `tools/list`;
   collect every tool's name, description and schema.
2. **Render** those schemas into the stable prefix of the prompt (lecture 21,
   component 2). Fifty servers can mean hundreds of tool descriptions and tens
   of thousands of prefix tokens, which is a context cost before the task
   starts; hosts increasingly load tool descriptions lazily or let the model
   search for a tool by name.
3. When the model emits a call, **validate** it against the schema, **check
   permissions** (which servers may write? which need approval each time?),
   ask the user if required, then send `tools/call`.
4. **Clip** the result and append it as a tool-result turn. Results are
   untrusted text (section 7).


The protocol standardizes steps 1 and 3. Steps 2 and 4 are still the harness's
job, and still where most of the quality lives.


## Agent-to-agent: A2A, and where the protocols stand in 2026

- MCP connects an agent to *tools*. **A2A** (Agent2Agent) connects an agent
  to *other agents* that keep their own memory, tools and logic private.
  Announced by Google in April 2025 as complementary to MCP
  ([Google 2025](https://developers.googleblog.com/en/a2a-a-new-era-of-agent-interoperability/)).
- **Discovery by Agent Card.** An agent publishes a JSON document at
  `/.well-known/agent-card.json` listing its endpoint, capabilities, required
  authentication and *skills*
  ([A2A agent discovery](https://a2a-protocol.org/latest/topics/agent-discovery/));
  a client agent reads the card and decides what to delegate.
- **The unit of work is a Task** with a lifecycle (submitted, working,
  input-required, completed, failed, ...); agents exchange **Messages** and
  return **Artifacts** ([A2A specification](https://a2a-protocol.org/latest/specification/)).
  A2A v1.0, the first stable specification, shipped in March 2026 with
  multi-protocol bindings, version negotiation, multi-tenancy and signed
  Agent Cards ([AAIF, Aug 2026](https://aaif.io/blog/a2a-joins-aaif)).
- **Both protocols left their vendors.** In December 2025 Anthropic donated
  MCP to the Linux Foundation's new **Agentic AI Foundation**, co-founded
  with OpenAI (which contributed `AGENTS.md`) and Block (goose)
  ([Anthropic 2025](https://www.anthropic.com/news/donating-the-model-context-protocol-and-establishing-of-the-agentic-ai-foundation));
  A2A joined it in August 2026. MCP now carries optional extensions (*Tasks*,
  *Skills over MCP*, *MCP Apps*;
  [MCP specification](https://modelcontextprotocol.io/specification/2026-07-28/)).
- **What did not change.** The model still reads a schema and emits JSON. A
  protocol makes tools *available*; it does not make the model *choose* them
  well, keep its context small, or resist a hostile tool result. Those are the
  next four sections.


# Memory and context management

Protocols decide what an agent can reach. Memory decides what it can hold. The
context window is the only memory the model reads directly, it is finite in
two senses, and it costs money on every call; everything else is a strategy
for keeping the right things in it.


## The context window is working memory

- The model's state between two of its own outputs is **exactly its context**:
  the system prompt, the tool schemas, the transcript so far, the latest tool
  result. Nothing else persists inside the model. An agent that "remembers"
  something has it in context, or has a way to put it back.
- It is finite twice. The hard limit is the window (128K to 1M tokens in 2026
  models). The soft limit is attention: performance on a fact degrades with
  the amount of other text around it long before the window is full (next
  slides).
- And it is paid for on every call. A 40-turn task re-sends its growing
  transcript 40 times; section 3's last slide does the arithmetic.
- So the engineering question is not "how do we fit everything in" but "what
  is the smallest set of tokens that lets the next step succeed".


## Context engineering, continued from lecture 13

Lecture 13 introduced **context engineering** as the discipline that includes
RAG: managing everything the model sees before it generates. Anthropic's
engineering note defines it as "curating and maintaining the optimal set of
tokens during LLM inference" and lists the techniques agents use
([Anthropic 2025](https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents)):

- **Compaction.** When the transcript nears the limit, summarize it (decisions
  made, files touched, open problems) and restart the context from the
  summary. Lossy by design; what is kept is a design choice.
- **Structured note-taking.** The agent writes a plan, progress and decisions
  to a file outside the context and re-reads it after compaction or a
  restart. Lecture 21's working-memory JSON is this.
- **Just-in-time retrieval.** Keep lightweight *identifiers* (paths, URLs,
  run ids) in context and fetch content only when needed, instead of
  preloading everything.
- **Sub-agent isolation.** Give a side task its own clean context and return
  only a condensed answer (lecture 21, component 6).


## Memory tiers

![Three tiers and the five operations that move tokens out of the window. Source: instructor figure.](figs/22-Agent-Infrastructure-and-Evaluation/memory-tiers.png)

The operating-system analogy is deliberate: most 2026 agent memory systems
descend from **MemGPT**, which framed the window as RAM and external storage
as disk with the model paging in and out through tool calls
([Packer et al. 2023](https://arxiv.org/abs/2310.08560)), or from the
generative-agents *memory stream* with periodic reflection
([Park et al. 2023](https://arxiv.org/abs/2304.03442)), plus a vector index.


## Retrieval-backed long-term memory, and its two failure modes

- **What goes in.** Past sessions' notes, user preferences, documents, code,
  an experiment ledger. Written by the agent (a `remember` tool), by a
  background summarizer, or by ingestion.
- **How it comes back.** The same pipeline as lecture 13's RAG: embed, index,
  retrieve top-$k$, insert into context. The query is now the agent's own
  current state rather than a user question, which makes relevance harder to
  judge.
- **Failure 1: retrieving the wrong thing.** A stale note from a previous
  task that contradicts the current one; a similar-looking but different
  file. The model treats retrieved text as true, so memory errors look like
  confident knowledge.
- **Failure 2: never retrieving.** The agent does not know what it does not
  remember, so it never asks. Mitigation: retrieve on every step with a tiny
  budget, and keep a short *index* of what memory contains inside the
  context.
- Evaluate memory the way you evaluate RAG (lecture 13's triad): was the
  right memory retrieved, was it used, did the answer improve.


## The cost picture: tokens per task

Lecture 21's symbols: a stable prefix of $P$ tokens (system prompt, tools,
workspace), each step appends $t$ tokens (the model's output plus the clipped
tool result), $K$ steps. Input tokens processed over the whole task:

$$
\text{no cache:}\quad \sum_{k=1}^{K}\big(P + (k-1)t\big) \;=\; KP + \frac{t\, K(K-1)}{2}.
$$

With a prefix cache, each token is **written** once (price $1.25\times$) and
**read** from cache on every later step (price $0.1\times$; lecture 21's
Anthropic ratios):

$$
W = P + (K-1)t, \qquad
R = (K-1)P + \frac{t\,(K-1)(K-2)}{2}, \qquad
\text{cost} = 1.25\,W + 0.1\,R .
$$


**Lecture 21's example in this form**, $P = 20{,}000$, $t = 1{,}000$, $K = 20$:
no cache $= 400{,}000 + 1{,}000 \times 190 = 590{,}000$ tokens;
$W = 39{,}000$, $R = 380{,}000 + 1{,}000 \times 171 = 551{,}000$, so cached cost
$= 48{,}750 + 55{,}100 = 103{,}850$ price-equivalent tokens, **about 5.7× less**,
the same total as lecture 21's step-by-step sum.
The quadratic term dominates, which is why *every* technique on the previous
two slides reduces $t$ or resets $K$, and why the cache is not optional.
Quizzable: redo this for other $P$, $t$, $K$ and price ratios.


Both formulas as a calculator, checked against a step-by-step sum so the
closed forms cannot drift from the definition. Change $P$, $t$, $K$ or the two
price multipliers and rerun; the second part answers the design question
"how much does halving $t$ (clipping tool output harder) buy, compared with
halving $K$ (compacting)?".

In [ ]:
P, t, K = 20_000, 1_000, 20      # prefix tokens, tokens appended per step, steps
write, read = 1.25, 0.10         # cache write and read prices, as multiples of plain input

def no_cache(P, t, K):
    return K * P + t * K * (K - 1) / 2

def cached(P, t, K, write=write, read=read):
    W = P + (K - 1) * t                          # every token is written once
    R = (K - 1) * P + t * (K - 1) * (K - 2) / 2  # and read on every later step
    return write * W + read * R, W, R

brute = sum(P + (k - 1) * t for k in range(1, K + 1))
cost, W, R = cached(P, t, K)
print(f"no cache: {no_cache(P, t, K):,.0f} tokens (step-by-step sum: {brute:,})")
print(f"W = {W:,}, R = {R:,.0f}: cached cost = {write * W:,.0f} + {read * R:,.0f} = {cost:,.0f}")
print(f"ratio: {no_cache(P, t, K) / cost:.2f}x")

print("\nwhat each lever buys (price-weighted input tokens, cached):")
for label, (p_, t_, k_) in {"baseline": (P, t, K), "halve P": (P // 2, t, K),
                             "halve t": (P, t // 2, K), "halve K": (P, t, K // 2)}.items():
    print(f"  {label:<9} {cached(p_, t_, k_)[0]:>9,.0f}   (no cache: {no_cache(p_, t_, k_):>9,.0f})")

## The cost curve

In [ ]:
#| code-fold: true
#| fig-cap: "Left: price-weighted input tokens against steps, without a cache (quadratic) and with one. Right: their ratio. Computed from the formulas on the previous slide."
#| fig-alt: "Two panels. Left: price-weighted input tokens in millions against steps from 1 to 60; the red no-cache curve rises quadratically to about 3.0 million, the green prefix-cache curve rises slowly to about 0.4 million; an annotation at K = 20 reads 0.59M versus 0.10M, 5.7 times cheaper. Right: the ratio no cache over cache rises from below 1 at one step to about 7.7 at 60 steps."
import numpy as np
import matplotlib.pyplot as plt

P, t = 20_000, 1_000             # prefix tokens, tokens appended per step
write, read = 1.25, 0.10         # cache write and read price multipliers
K = np.arange(1, 61)

no_cache = K * P + t * K * (K - 1) / 2
W = P + (K - 1) * t
R = (K - 1) * P + t * (K - 1) * (K - 2) / 2
cached = write * W + read * R

fig, axes = plt.subplots(1, 2, figsize=(10, 3.9))
ax = axes[0]
ax.plot(K, no_cache / 1e6, color="#e34948", lw=2.2, label="no cache: $KP + tK(K-1)/2$")
ax.plot(K, (W + R) / 1e6, color="#8a8984", lw=1, ls=":", label="writes + reads (same total)")
ax.plot(K, cached / 1e6, color="#008300", lw=2.2, label="prefix cache: $1.25\\,W + 0.1\\,R$")
ax.set_xlabel("steps in the task, $K$")
ax.set_ylabel("input tokens, price-weighted (millions)")
ax.set_title("$P$ = 20k prefix, $t$ = 1k per step", fontsize=10)
ax.legend(fontsize=8.5, loc="upper left", frameon=False)
i = 19                           # K = 20
ax.axvline(K[i], color="#8a8984", lw=0.8, ls="--")
ax.annotate(f"K = {K[i]}: {no_cache[i] / 1e6:.2f}M vs {cached[i] / 1e6:.2f}M\n"
            f"({no_cache[i] / cached[i]:.1f}x cheaper)",
            xy=(K[i], cached[i] / 1e6), xytext=(24, 1.3), fontsize=8.5,
            arrowprops=dict(arrowstyle="->", color="#8a8984"))

ax = axes[1]
ax.plot(K, no_cache / cached, color="#2a78d6", lw=2.2)
ax.axhline(1 / read, color="#8a8984", lw=0.8, ls="--")
ax.text(60, 1 / read - 0.2, "limit $1/0.1 = 10$", ha="right", va="top", fontsize=8.5, color="#8a8984")
ax.set_xlabel("steps in the task, $K$")
ax.set_ylabel("cost ratio, no cache / cache")
ax.set_title("the saving grows with the task length", fontsize=10)
ax.set_ylim(0, 10.5)
for ax in axes:
    ax.grid(color="#dcdcd8", lw=0.6)
    ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()

## Failure mode: lost in the middle, and context rot

- **Lost in the middle.** On multi-document QA and key-value retrieval,
  accuracy is a U-shaped function of where the relevant passage sits:
  highest at the start or end of the context, lowest in the middle, even for
  models built for long context
  ([Liu et al. 2023](https://arxiv.org/abs/2307.03172)). A transcript that
  buries the one relevant tool result under twenty later ones is this
  experiment run by accident.
- **Context rot.** Across 18 models, performance on simple tasks falls as
  input length grows, falls faster when the question and the needle match
  semantically rather than lexically, and a single distractor measurably
  hurts ([Hong et al. 2025](https://www.trychroma.com/research/context-rot)).
  The window being 1M tokens does not mean the 1M tokens are equally usable.
- Both are reasons the harness techniques exist: a compacted transcript puts
  the summary at the *start*; just-in-time retrieval puts the needed chunk at
  the *end*; both keep the middle short.
- The same results apply to **tool descriptions**: hundreds of schemas in the
  prefix are a middle the model must search through on every turn.
- **Tool-output flooding** is the agent's version: one `cat` of a log file,
  one unfiltered search or one `SELECT *` pushes the task description far
  away, and a remote server's output size is not under the harness's control.
  Bound a result's size in the *tool* (a per-result token cap, pagination, a
  summary plus a handle to the full result), not in the model's judgement in
  calling it.


# Multi-agent patterns

Sub-agent isolation was the last memory technique; taken further it becomes a
system of several agents. This section is about the handful of patterns that
recur, the evidence for when they help, and a careful study of how they fail.


## Workflows and agents

Anthropic's December 2024 note draws a line worth keeping
([Anthropic 2024](https://www.anthropic.com/engineering/building-effective-agents)):

- **Workflows**: "LLMs and tools are orchestrated through predefined code
  paths". The control flow is yours; the model fills in steps.
- **Agents**: "LLMs dynamically direct their own processes and tool usage".
  The control flow is the model's.
- The advice: start with a single call; add a workflow when a fixed
  decomposition is obvious; use an agent only for "open-ended problems where
  it is difficult or impossible to predict the required number of steps".
  Agents cost more and compound errors.
- Most "multi-agent systems" in production are workflows whose steps happen
  to be model calls with their own prompts. That is fine, and it is easier to
  test.


## Four patterns

![Orchestrator and workers; parallel fan-out by sectioning or voting; handoffs between specialists; a generator-verifier loop. Source: instructor figure after Anthropic (2024).](figs/22-Agent-Infrastructure-and-Evaluation/multi-agent-patterns.png)

- **Orchestrator–workers**: one agent plans and delegates; workers run with
  clean contexts; the orchestrator synthesizes. Context isolation is the point.
- **Parallel fan-out**: independent sub-tasks at once (*sectioning*), or the
  same task $N$ times with a vote (*voting*; lecture 21's self-consistency).
- **Handoffs**: a conversation passes between specialists with their own
  tools and prompts; one is active at a time.
- **Generator–verifier**: one agent drafts, another grades or critiques, loop
  until pass. Pays off when checking is cheaper than producing.


## When more agents help

Anthropic's research system is an orchestrator-workers design: a lead agent
decomposes a query and spawns parallel sub-agents that search and return
condensed findings
([Anthropic 2025](https://www.anthropic.com/engineering/multi-agent-research-system)).
What they measured:

- The multi-agent system (Opus 4 lead, Sonnet 4 workers) beat a single Opus 4
  agent by **90.2%** on their internal research eval, with breadth-first
  queries (many independent directions) gaining most.
- In their analysis of the single-agent baseline, **token usage alone
  explained 80% of the variance** in score; number of tool calls and model
  choice most of the rest. Multi-agent is, first, a way to spend more tokens
  in parallel without one context rotting.
- The cost: agents use about **4×** the tokens of a chat, multi-agent about
  **15×**. It pays only for high-value tasks.
- Where it does not help, in their words: tasks needing shared context,
  heavy interdependence, or real-time coordination, and "most coding tasks",
  which have less parallelizable work than research.


## When they fail: MAST

![The Multi-Agent System Failure Taxonomy: 14 failure modes in three categories, placed by the conversation stage at which they occur, with their frequency in the annotated traces. Figure 1 of Cemri et al., 2025.](figs/22-Agent-Infrastructure-and-Evaluation/mast-taxonomy-cemri-fig1.png)

- [Cemri et al. 2025](https://arxiv.org/abs/2503.13657) annotated over 1,600
  execution traces from 7 multi-agent frameworks and found that failures
  cluster into **system design issues** (44%), **inter-agent misalignment**
  (32%) and **task verification** (24%).
- The single most common modes are mundane: step repetition, a reasoning
  action mismatch (the agent says one thing and does another), disobeying the
  task specification, not knowing when to stop.
- Their reading: these are *system design* failures more than model failures,
  analogous to the organizational failures studied in high-reliability
  organizations, and prompt tweaks alone fixed only some of them.


## Rules that follow

- **Specify the task, the role, and the stopping condition** in writing, to
  each agent. Three of the five design-issue modes are a missing
  specification.
- **Verify at the end, with something other than the agent that did the
  work.** A quarter of MAST's failures are verification failures; a generator
  that grades itself is the worst case.
- **Budget tokens per agent and per task** and stop at the budget. Step
  repetition is both a failure mode and a cost.
- **Prefer a workflow when the decomposition is known.** Fixed code paths
  cannot derail, withhold information, or reset the conversation.
- **Hold multi-agent to a cost-matched baseline**: the same model, single
  agent, given 15× the budget by other means (more samples, longer context,
  retrieval). Anthropic's 90.2% is against a single agent, not against a
  single agent at equal cost.


## Research-agent harnesses: the loop applied to ML

Lecture 21 covers autoresearch and OpenResearch; seen through this lecture
they are harness designs (one agent, one tool, one verifiable number, with an
experiment ledger as section 3's retrieval-backed memory).

- Where these loops fail is section 5's list: hacking the metric, overfitting
  the validation split, and changes that only help at a 5-minute budget. The
  midterm project asks you to find those by re-running with seeds.


# Agentic RL

Everything so far is engineering around a fixed model. Lecture 14 had two
slides on training the model *for* the loop; this section says what
changes when the trajectory contains tool calls, and where the reward comes
from.


## What changes from lecture 14

| | Lecture 14 (RLVR on a response) | Agentic RL |
|---|---|---|
| Episode | one prompt, one response | many turns: think, call, observe, repeat |
| State | the prefix | the whole transcript, including tool outputs |
| Actions | every generated token | only the policy's tokens; tool outputs are **masked** from the loss |
| Reward | terminal, from a checker | terminal, from the *environment's* final state, after real side effects |
| Horizon | hundreds to thousands of tokens | tens of calls, tens of thousands of tokens, minutes of wall clock |
| Observability | full | partial: the environment has state the transcript does not show |

The survey framing: conventional LLM-RL is a "degenerate single-step MDP";
agentic RL is a "temporally extended, partially observable MDP"
([Zhang et al. 2025](https://arxiv.org/abs/2509.02547)). The estimator is
unchanged (lecture 14's policy gradient with a group baseline); **masking**
matters because a tool output is the environment's transition, which the
derivation showed carries no gradient, and because back-propagating through
retrieved text makes training unstable, as Search-R1 found when it introduced
retrieved-token masking ([Jin et al. 2025](https://arxiv.org/abs/2503.09516)).
VerlTool packages the table: multi-turn trajectories with observation tokens,
a uniform tool API (code execution, search, SQL, vision), and tool execution
run asynchronously with generation for a near-2× rollout speedup
([Jiang et al. 2025](https://arxiv.org/abs/2509.01055)); lecture 14's
trainer-consuming-a-queue picture is the result.


## Credit assignment over a long horizon

- One terminal reward over 30 tool calls says nothing about *which* call was
  the mistake. Group-relative advantages (GRPO) compare whole trajectories,
  which works but is sample-hungry: the signal per token shrinks with
  trajectory length.
- **Branch where it matters.** ARPO observed that token entropy spikes right
  after a tool result arrives (the model is uncertain what to do with new
  information), and branches extra rollouts at those steps instead of only at
  the start, with step-level advantage attribution; reported gains on 13
  benchmarks at about half the tool-call budget
  ([Dong et al. 2025](https://arxiv.org/abs/2507.19849)).
- **Process rewards** score intermediate steps. For math, step-level human
  labels (PRM800K) trained a verifier that beat outcome supervision
  ([Lightman et al. 2023](https://arxiv.org/abs/2305.20050)); for agents,
  per-step judgements of a transcript play the same role but inherit every
  judge bias of lecture 13. Section 6 returns to this as *trajectory grading*.
- The honest summary: long-horizon credit assignment is the open problem of
  agentic RL in 2026, and most working systems still use a terminal reward
  with many samples.


## Environment and verifier design

- **The environment is a sandbox per rollout**: a container with the repo, a
  browser, a database. It must be *resettable* (same start state every
  sample), *isolated* (one rollout's side effects cannot leak into another's),
  and *fast* (the rollout, not the gradient step, is the bottleneck).
- **The verifier reads the environment, not the transcript.** For code, run
  the tests; for a customer-service agent, compare the database's final state
  with the annotated goal state, which is how τ-bench grades
  ([Yao et al. 2024](https://arxiv.org/abs/2406.12045)); for a research
  agent, the metric on a held-out split.
- A verifier that reads the transcript ("did the agent say it ran the tests")
  is a verifier that will be lied to.


## Where the reward comes from, by domain

| Agent | Reward | Source | Weakness |
|---|---|---|---|
| Coding | hidden tests pass | SWE-bench-style tasks; or patch similarity to the merged fix, which SWE-RL used to train a 70B model to 41.0% on SWE-bench Verified ([Wei et al. 2025](https://arxiv.org/abs/2502.18449)) | tests can be deleted, mocked or special-cased |
| Search / browsing | exact-match answer | QA datasets with retrieval (Search-R1) | the answer can be found in a leaked page |
| Tool-use / customer service | final database state equals goal state | τ-bench | the state check misses what the agent said to the user |
| ML research | validation metric after a fixed budget | autoresearch's `val_bpb` | overfits the validation split; gains that vanish at scale |
| Anything else | an LLM judge on the transcript | rubric + judge (lecture 13) | the judge is a reward model, and lecture 14's over-optimization applies |

**Reward hacking is now an agent behaviour.** Hack-verifiable terminal
environments embed detectable shortcuts into tasks so that hacks are caught
automatically rather than by reading transcripts; prompting against known
hacks does not prevent the unknown ones
([Roth et al. 2026](https://arxiv.org/abs/2608.22103)). Lecture 14's checks
(held-out accuracy, a random-reward control, reading samples) apply unchanged,
and "read samples" now means reading trajectories.


# Evaluating agents

A verifier for training and a grader for evaluation are the same artifact used
for different purposes, and this section is about the second use. Lecture 13
built an eval for a model's single response; an agent's output is a trajectory
with side effects and a cost, and that changes what to measure.


## Why pass@1 on a benchmark is not enough

- **A trajectory, not a response.** The same final state can be reached by a
  clean run or by one that deleted the tests, and a benchmark that checks only
  the end cannot tell. Lecture 13's grader saw everything it graded; an
  agent's grader does not.
- **Variance across runs.** An agent that solves a task 6 times in 10 scores
  60% on pass@1 and is useless as a deployed service. Reliability is a
  separate axis (pass^k, below).
- **Cost.** Two agents at the same accuracy can differ 10× in tokens. "AI
  Agents That Matter" argued that accuracy-only leaderboards reward
  needlessly complex and expensive agents and called for cost-controlled
  evaluation on a Pareto frontier
  ([Kapoor et al. 2024](https://arxiv.org/abs/2407.01502)).
- **Contamination and overfitting** are worse than for static QA: the task
  repositories are public, and an agent harness can be tuned to a suite's
  quirks. The same paper found many agent benchmarks have inadequate holdout
  sets, and sometimes none at all (Kapoor et al.).
- **Harness versus model.** Lecture 21's claim that the harness matters as
  much as the model means a leaderboard number is a *system* number; the same
  model in a different harness scores differently.


## Task suites

| Suite | What a task is | Grader | At release |
|---|---|---|---|
| [**SWE-bench**](https://arxiv.org/abs/2310.06770) (Jimenez et al. 2023) | 2,294 real GitHub issues from 12 Python repos; produce a patch | the repo's hidden tests | Claude 2 resolved 1.96% |
| **SWE-bench Verified** ([OpenAI 2024](https://openai.com/index/introducing-swe-bench-verified/)) | 500 of them screened by 93 developers for clear specification and correct tests | same | GPT-4o 33.2% |
| [**WebArena**](https://arxiv.org/abs/2307.13854) (Zhou et al. 2023) | long-horizon tasks on self-hosted e-commerce, forum, code and CMS sites | programmatic check of the end state | GPT-4 agent 14.4%; humans 78.2% |
| [**τ-bench**](https://arxiv.org/abs/2406.12045) (Yao et al. 2024) | retail and airline customer service with a *simulated user*, APIs and policy rules | final database state vs. goal state; pass^k | GPT-4o 61.2% (retail), 35.2% (airline) pass^1 |
| [**GAIA**](https://arxiv.org/abs/2311.12983) (Mialon et al. 2023) | 466 questions needing browsing, tools and multi-modal reasoning; a short exact answer | exact match | humans 92%; GPT-4 with plugins 15% |
| [**Terminal-Bench**](https://arxiv.org/abs/2601.11868) (Merrill et al. 2026) | 89 hard tasks in containerized terminals, each with a human solution and tests (v2.0); later versions add tasks and report cost | tests; resolved rate with tokens and cost | frontier agents under 65% |

Every one of these starts far from the human number and saturates within a
few years; SWE-bench Verified's top entries approached 90% by mid-2026
(third-party aggregator; confirm on [swebench.com](https://www.swebench.com/)).
<!-- TODO(verify): the "above 90% by mid-2026" figure is from a third-party aggregator (benchlm.ai); confirm on swebench.com before the term -->
The suite's useful life is lecture 13's window between "can't" and "has seen
it", and the METR horizon (on the cost slide) is one way to keep measuring
after that.


## What a SWE-bench task looks like

![A task instance: the issue text and a repository snapshot are the input; the gold patch is hidden; the test patch's new assertion must pass after the agent's patch is applied. Figure 1 of Jimenez et al., 2023.](figs/22-Agent-Infrastructure-and-Evaluation/swe-bench-task-jimenez-fig1.png)


## Trajectory grading

- **Outcome grading**: did the end state satisfy the check. Cheap, objective,
  blind to *how*. It is the right primary metric when the check is strong
  (tests, database state) and the environment is sandboxed.
- **Process grading**: score the steps. Did the agent run the tests before
  claiming success? Did it read the file it edited? Did it follow the policy
  (τ-bench's rules), ask before an irreversible action, stay inside the
  allowed tools? Process reward models do this for training
  ([Lightman et al. 2023](https://arxiv.org/abs/2305.20050)); for evaluation
  it is usually a **rubric applied to the transcript**.
- **LLM judges on transcripts** inherit every bias of lecture 13: verbosity
  (a long transcript looks thorough), position, self-preference, and a new
  one, **narration bias**: the judge believes what the agent *says* it did.
  Give the judge the tool results, not only the agent's messages, and check
  agreement with a human on 50 trajectories (lecture 13's $\kappa$).
- **Rule-based process checks** are better where they exist: count tool
  calls, detect test deletion, diff the allowed-file set, flag network calls.
  MAST's taxonomy is a checklist of what to look for.


## Reliability: pass^k

τ-bench's reliability metric ([Yao et al. 2024](https://arxiv.org/abs/2406.12045)):
run each task $n$ times, count $c$ successes, and estimate the chance that
**all $k$** independent attempts succeed:

$$
\text{pass}^{\wedge}k \;=\; \mathbb{E}_{\text{task}}\!\left[\frac{\binom{c}{k}}{\binom{n}{k}}\right].
$$

- Why that ratio: if $k$ of the $n$ trials are drawn without replacement, the
  probability that all $k$ are successes is $\binom{c}{k}/\binom{n}{k}$.
  Under the model that a task's trials succeed i.i.d. with probability $p$,
  this is an unbiased estimate of $p^k$. So $\text{pass}^{\wedge}k$ estimates
  $\mathbb{E}_{\text{task}}[p^k]$.
- Contrast **pass@k** from the Codex paper
  ([Chen et al. 2021](https://arxiv.org/abs/2107.03374)),
  $\mathbb{E}\big[1 - \binom{n-c}{k}/\binom{n}{k}\big]$, the chance that *at
  least one* of $k$ attempts succeeds, $\mathbb{E}[1 - (1-p)^k]$. pass@k rises
  with $k$ (good for search with a verifier); pass^k falls (what a user who
  gets one attempt experiences).
- **Worked example**: $n = 8$, $c = 6$, $k = 2$:
  $\binom{6}{2}/\binom{8}{2} = 15/28 = 0.536$, while pass@2
  $= 1 - \binom{2}{2}/\binom{8}{2} = 27/28$. τ-bench's headline: GPT-4o at 61%
  pass^1 on retail fell below 25% at pass^8.


## What the slope of pass^k tells you

![Left: pass^k (solid) and pass@k (dashed) when every task has the same success probability. Right: three agents with the same pass^1 of 0.8 and different failure structure. Source: instructor figure from the formulas.](figs/22-Agent-Infrastructure-and-Evaluation/pass-hat-k-curves.png)

- Since $\mathbb{E}[p^k] \ge (\mathbb{E}[p])^k$ (Jensen; $p^k$ is convex), a
  **flat** pass^k curve means the failures are *task-level* (some tasks the
  agent never solves), a **steep** one means they are *run-level* (the agent
  is flaky on tasks it can solve). Same pass^1, different fix: harder tasks
  need capability, flakiness needs determinism, better tools, or a verifier
  loop.
- Report $n$: with $n = 8$ the estimate of $p^8$ comes from one trial per
  task and is all noise. Lecture 13's standard-error reasoning applies
  per task.
- Quizzable: given $n$, $c$, $k$, compute pass^k and pass@k; given two
  curves, say which agent has task-level and which has run-level failures.


The estimator on the worked example, then a simulation of the slide's
argument. Three agents with the same pass^1 of 0.8 each attempt 2,000 tasks
$n = 16$ times: one with every task at $p = 0.8$ (all failures run-level),
one whose per-task $p$ follows a Beta(2, 0.5) distribution (mean 0.8, most
tasks easy, a tail of hard ones), and one that solves 80% of tasks always and
the rest never (all failures task-level). The dots are the
$\binom{c}{k}/\binom{n}{k}$ estimator averaged over tasks; the lines are the
true $\mathbb{E}[p^k]$, which the dots match up to sampling noise; the
dashed line is $(\mathbb{E}p)^k = 0.8^k$, which
only the uniform agent attains. Every other agent sits above it, as Jensen
says.

In [ ]:
#| code-fold: true
#| fig-cap: "pass^k against $k$ for three agents with the same pass^1 = 0.8: the unbiased estimator from 2,000 simulated tasks with $n = 16$ runs each (dots), the true $\\mathbb{E}[p^k]$ (lines), and $0.8^k$ (dashed)."
#| fig-alt: "pass^k from 0 to 1 against k from 1 to 16. All three agents start at 0.8. The uniform agent falls along the dashed 0.8 to the k curve to about 0.03. The Beta-distributed agent falls more slowly, to about 0.3 at k = 16. The task-level agent stays flat at 0.8. Simulated dots lie on the lines."
import numpy as np
import matplotlib.pyplot as plt
from fractions import Fraction
from math import comb

# The slide's worked example
n, c, k = 8, 6, 2
pass_hat = Fraction(comb(c, k), comb(n, k))
pass_at = 1 - Fraction(comb(n - c, k), comb(n, k))
print(f"n = {n}, c = {c}, k = {k}: pass^k = {pass_hat} = {float(pass_hat):.3f}, "
      f"pass@k = {pass_at} = {float(pass_at):.3f}")

# Simulation: same mean success, different spread of per-task p
rng = np.random.default_rng(0)
n_tasks, n = 2000, 16
ks = np.arange(1, n + 1)
a, b = 2.0, 0.5                                   # Beta(a, b), mean a / (a + b) = 0.8
agents = {
    "uniform: every task at p = 0.8": (np.full(n_tasks, 0.8),
                                       0.8 ** ks),
    "heterogeneous: p ~ Beta(2, 0.5)": (rng.beta(a, b, n_tasks),
                                        np.cumprod((a + ks - 1) / (a + b + ks - 1))),
    "task-level: 80% always, 20% never": ((np.arange(n_tasks) < 0.8 * n_tasks).astype(float),
                                          np.full(n, 0.8)),
}

def pass_hat_k(c, n, k):
    return np.array([comb(int(ci), k) / comb(n, k) for ci in c]).mean()

fig, ax = plt.subplots(figsize=(7.5, 4))
for (label, (p, exact)), color in zip(agents.items(), ["#2a78d6", "#e87ba4", "#4a3aa7"]):
    successes = rng.binomial(n, p)                # c for each task
    est = [pass_hat_k(successes, n, k) for k in ks]
    ax.plot(ks, exact, color=color, lw=2, label=label)
    ax.plot(ks, est, "o", color=color, ms=4)
    print(f"{label:<36} pass^1 {est[0]:.3f}  pass^4 {est[3]:.3f} (true {exact[3]:.3f})  "
          f"pass^16 {est[15]:.3f} (true {exact[15]:.3f})")
ax.plot(ks, 0.8 ** ks, color="#8a8984", ls="--", lw=1, label="$(\\mathbb{E}p)^k = 0.8^k$")
ax.set_xlabel("$k$")
ax.set_ylabel("pass^k")
ax.set_ylim(0, 1.02)
ax.grid(color="#dcdcd8", lw=0.6)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(fontsize=8.5, frameon=False, loc="lower left")
fig.tight_layout()
plt.show()

## Cost per task, and the frontier

- Report, per task: **resolved** (outcome), **tokens in and out** (and cache
  hits), **dollars** at stated prices, **wall-clock time**, **tool calls**.
  Terminal-Bench's leaderboard lists cost and tokens next to the resolve
  rate; ARC-AGI plots score against cost per task (lecture 13).
- Plot **accuracy against cost** and look at the Pareto frontier. A harness
  that adds a verifier loop may raise accuracy 5 points at 3× the cost; that
  is a point on the frontier, not an improvement, until the cost is in the
  table ([Kapoor et al. 2024](https://arxiv.org/abs/2407.01502)).
- Cost is not stable: it depends on prompt caching (section 3's 5.7×),
  model prices, and the max-turns budget. State all three.
- Lecture 15's arithmetic connects: tokens per task × tasks per day × price
  per token is the serving bill, and agentic workloads are prefill-heavy
  (long prompts, short outputs), which is what makes prefix caching the
  biggest lever.
- **A trend line that does not saturate: the METR task horizon** (the figure
  is in lecture 1). For each model, fit a logistic curve of success against
  the human time a task takes; the **50% horizon** is the task length at
  which it crosses 0.5, and it has doubled about every seven months since
  2019 ([Kwa et al. 2025](https://arxiv.org/abs/2503.14499)). The main suite
  is RE-Bench, HCAST and 66 short SWAA tasks with METR-measured human times;
  SWE-bench Verified is used only as an external-validity check with the
  annotators' time estimates (§4.1). Caveats: clean, self-contained software
  tasks, and the 80% horizon is about five times shorter.


## Building the final-project agent eval

Lecture 13's five-part recipe, with the agent-specific additions:

1. **Tasks with an environment.** 30 to 100 tasks, each a *reset-able*
   start state (a repo at a commit, a database snapshot, a folder of files)
   and a programmatic end-state check. Hold out a third you never tune on.
2. **A grader that reads the environment**, not the transcript: tests, state
   diff, exact answer. Add two or three **process checks** (ran the tests?
   stayed in the allowed files? no network?).
3. **A cost budget per task** (max turns, max tokens) that the harness
   enforces, and the same budget for every system you compare.
4. **$n \ge 5$ runs per task**; report pass^1 with a standard error, pass^k
   for the $k$ your use case implies, and the cost columns from two slides
   ago.
5. **An agreement check** on any LLM-judged rubric: 50 trajectories graded
   by you, Cohen's $\kappa$ against the judge, fix the rubric below 0.6.


Then compare *harness changes* at a fixed model before comparing models. That
ordering is lecture 21's thesis, tested.


# Safety and failure modes

Every section so far gave the agent more reach: more tools, more memory, more
agents, more autonomy from training. The last section is about the failure
that reach creates, and the small set of structural defenses that work.


## Prompt injection via tool results

- The model cannot distinguish instructions from data: both are tokens in the
  same context. **Indirect prompt injection** places instructions in data the
  agent will *retrieve* (a web page, an email, a file, a tool description) so
  that an attacker with no access to the prompt still steers the agent.
  Demonstrated in 2023 against Bing Chat (GPT-4-powered) and code-completion
  engines such as GitHub Copilot
  ([Greshake et al. 2023](https://arxiv.org/abs/2302.12173)).
- Section 2 made this worse by construction: every MCP tool result and every
  resource is text from a server the host may not control, and the spec says
  tool descriptions themselves "should be considered untrusted".
- **Measured**: AgentDojo built 97 realistic tasks (email, banking, travel)
  with 629 injection test cases; attacks succeeded on a meaningful fraction
  and defenses that stopped them cost utility
  ([Debenedetti et al. 2024](https://arxiv.org/abs/2406.13352)).
- There is no training-time fix in 2026 that is reliable enough to build on.
  Treat injection as a given and design so that a successful injection cannot
  do much.


## The lethal trifecta

![Three capabilities; the danger is their intersection. Source: instructor figure after Willison (2025).](figs/22-Agent-Infrastructure-and-Evaluation/lethal-trifecta-venn.png)

- [Willison 2025](https://simonwillison.net/2025/Jun/16/the-lethal-trifecta/):
  an agent with **access to private data**, **exposure to untrusted
  content** and **a way to communicate externally** can be made to exfiltrate
  the data by an instruction hidden in the content. Each capability alone is
  fine; the combination is the vulnerability.
- The defense is *structural*: remove one. A browsing agent with no private
  data; a data agent with no network; an email agent that cannot send
  without a human click.
- Meta's **Rule of Two** is the same statement as a policy: an agent session
  may have at most two of *untrustworthy inputs*, *sensitive data*, *external
  actions*; a task that needs all three runs with human approval, not
  autonomously ([Meta 2025](https://ai.meta.com/blog/practical-ai-agent-security/)).
- Quizzable: given an agent's tool list, name which of the three it has and
  what to remove.


## Permission scoping, sandboxing, checkpoints

- **Least privilege per tool.** Read-only by default; write and network
  tools on an allowlist; separate credentials per server so a compromised one
  cannot reach the others. Lecture 21's subagents are read-only for this
  reason.
- **Sandbox the execution**, not the model: a container or VM per task with
  no network except allowlisted hosts, a copy of the data rather than the
  data, a filesystem that is thrown away. The RL environments of section 5
  already have this shape; reuse it in deployment.
- **Human-in-the-loop checkpoints** at *irreversible* actions: sending,
  paying, deleting, pushing to main, anything outside the sandbox. Approval
  fatigue is real, so gate the irreversible steps and not every file read.
- **Budgets as a safety tool**: a max-turns and max-cost limit stops a
  derailed agent (MAST's step repetition) as well as a hijacked one.
- **Protocol-level hooks**: MCP's consent principles put the approval in the
  host; A2A's signed Agent Cards and declared security schemes decide which
  agents may be delegated to at all.


## Logging for audit

- Keep the **full trajectory**: every prompt as sent (including the tool
  schemas and cache boundaries), every model output, every tool call with
  arguments, every tool result before clipping, every approval. Lecture 21's
  append-only JSONL transcript is the minimum.
- It serves four purposes at once: **debugging** (why did it do that),
  **evaluation** (section 6's process checks run on logs), **training data**
  (section 5's trajectories), and **incident response** (what did the injected
  page make it do, and what left the sandbox).
- Use standard tracing: MCP now points implementers to OpenTelemetry rather
  than its own logging primitive
  ([deprecated features registry](https://modelcontextprotocol.io/specification/2026-07-28/deprecated)),
  and the agent frameworks emit spans per model call and per tool call.
- Logs contain the private data; they are themselves inside the trifecta.
  Redact and restrict them like the data they contain.


# Summary

One line per idea; the recap deck uses these.


## Summary

- A tool call is structured output against a JSON Schema; models learn it by SFT on tool-call traces, and constrained decoding guarantees syntax but not choice.
- MCP turns $N \times M$ connectors into $N + M$: hosts run the model, clients connect, servers expose tools, resources and prompts over JSON-RPC via stdio or Streamable HTTP; the host renders schemas into the prompt and holds the consent gate.
- A2A connects agents to agents: an Agent Card at a well-known URL, Tasks with a lifecycle, Messages, Artifacts; MCP and A2A are under the Linux Foundation's Agentic AI Foundation since 2025–26.
- The context window is the agent's only working memory, finite in tokens and in attention, and paid for on every call; compaction, notes, just-in-time retrieval and subagent isolation keep it small.
- Memory tiers: window, scratchpad files, retrieval-backed long-term store (MemGPT, generative agents); retrieval can fetch the wrong thing or nothing.
- Tokens per task without a cache are $KP + tK(K-1)/2$, quadratic in steps; with a prefix cache, $1.25W + 0.1R$ with $W = P + (K-1)t$ and $R = (K-1)P + t(K-1)(K-2)/2$; lecture 21's example ($P = 20$k, $t = 1$k, $K = 20$) gives 590,000 versus 103,850, about 5.7× less.
- Lost in the middle (U-shaped accuracy by position) and context rot (degradation with length, 18 models) mean a long window is not an equally usable window; tool-output flooding is the agent's version.
- Workflows have fixed control flow; agents direct their own. Patterns: orchestrator–workers, parallel fan-out (sectioning, voting), handoffs, generator–verifier.
- Multi-agent helps for parallelizable, breadth-first, high-value tasks (Anthropic: +90.2% at about 15× the tokens of chat) and fails by specification, inter-agent misalignment and verification (MAST: 14 modes, 1,600+ traces); compare against a cost-matched single agent.
- Research-agent harnesses (lecture 21's autoresearch and OpenResearch) are the loop applied to ML, and fail by hacking the metric, overfitting the validation split, and gains that only hold at a small budget.
- Agentic RL keeps lecture 14's estimator over multi-turn trajectories in a POMDP; tool outputs are masked, the reward comes from the environment's end state, and long-horizon credit assignment (ARPO's entropy-triggered branching, process rewards) is the open problem; rewards come from tests, state checks, metrics or judges, each gameable.
- pass@1 is not enough: trajectories have process, variance and cost. Suites: SWE-bench and Verified, WebArena, τ-bench, GAIA, Terminal-Bench; each saturates.
- pass^k $= \mathbb{E}[\binom{c}{k}/\binom{n}{k}]$ estimates $\mathbb{E}[p^k]$, the chance all $k$ attempts succeed; pass@k $= \mathbb{E}[1-\binom{n-c}{k}/\binom{n}{k}]$ is at-least-one; a flat pass^k curve means task-level failures, a steep one run-level.
- Report accuracy against cost per task on a Pareto frontier; the METR 50% horizon (doubling about every seven months) is a trend instrument that does not saturate.
- A project agent eval: reset-able tasks with an environment check, process checks, an enforced budget, $n \ge 5$ runs, a judge-agreement check.
- Indirect prompt injection arrives through tool results and is unsolved; the lethal trifecta (private data, untrusted content, exfiltration channel) is the structural rule: remove one, or require a human.
- Least privilege per tool, sandboxed execution, approval at irreversible actions, budgets, and a full trajectory log that is itself protected.

## Discussion questions

1. Your lab wants an agent that reads the group's experiment ledger, searches the web for related work, and posts summaries to Slack. Which trifecta capabilities does it have, and what is the least damaging one to remove?
2. Two coding agents both score 70% pass^1 on your task suite. Agent A's pass^5 is 65%; agent B's is 20%. Which one do you deploy, and what do you try to fix in the other?
3. A protocol makes thousands of tools available. Propose a mechanism for a harness to decide which 20 tool schemas go into the prompt for a given task, and how you would evaluate it.
4. Anthropic reports a multi-agent system beating a single agent by 90% while using many times the tokens. Design the cost-matched baseline that would tell you whether the *architecture* or the *budget* did the work.
5. For the midterm project's autoresearch loop: which of MAST's failure modes can occur with a single agent, and which reward hacks would the `val_bpb` metric not catch?

## Further reading

- [MCP specification](https://modelcontextprotocol.io/specification/2026-07-28/) and the [python-sdk README](https://github.com/modelcontextprotocol/python-sdk): read the architecture and server-features pages, then build the twenty-line server.
- [Anthropic, *Building effective agents* (2024)](https://www.anthropic.com/engineering/building-effective-agents) and [*Effective context engineering for AI agents* (2025)](https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents): the two short notes this lecture's sections 3 and 4 are organized around.
- [Cemri et al., *Why Do Multi-Agent LLM Systems Fail?* (2025)](https://arxiv.org/abs/2503.13657): the taxonomy, and the case studies of interventions that did and did not help.
- [Yao et al., *τ-bench* (2024)](https://arxiv.org/abs/2406.12045) and [Kapoor et al., *AI Agents That Matter* (2024)](https://arxiv.org/abs/2407.01502): pass^k, and cost-controlled evaluation.
- [Zhang et al., *The Landscape of Agentic Reinforcement Learning for LLMs* (2025)](https://arxiv.org/abs/2509.02547): the survey; section on environments and benchmarks is a map of section 5.
- [Greshake et al. (2023)](https://arxiv.org/abs/2302.12173), [Debenedetti et al., *AgentDojo* (2024)](https://arxiv.org/abs/2406.13352) and [Willison, *The lethal trifecta* (2025)](https://simonwillison.net/2025/Jun/16/the-lethal-trifecta/): the attack, the measurement, and the rule.
- [Kwa et al., *Measuring AI Ability to Complete Long Tasks* (2025)](https://arxiv.org/abs/2503.14499): the horizon method and its caveats, with METR's updated plot at [metr.org](https://metr.org/).